# Order Flow Imbalance on BTCUSDT

Microstructure seed model. Pulls one day of BTCUSDT aggregated trades, aggregates to 1-minute bars with order-flow-imbalance as a per-bar feature, and trades a smoothed-OFI threshold rule via the event-driven backtest engine.

See [`README.md`](README.md) for the writeup and [`backtest.py`](backtest.py) for the reproducible CLI version.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from backtest import (
    BAR_SECONDS,
    END,
    ENTRY_THRESHOLD,
    PERIODS_PER_YEAR,
    SMOOTH_WINDOW,
    START,
    SYMBOL,
    RollingOFIStrategy,
)

from tradinglib.backtest import bars_from_dataframe, run_event_backtest
from tradinglib.features.microstructure import aggregate_to_bars
from tradinglib.loaders.crypto.binance_trades import load_trades

## Load tick data and aggregate

In [ ]:
trades = load_trades(SYMBOL, START, END)
print(f"{len(trades):,} aggTrades")
bars_df = aggregate_to_bars(trades, bar_seconds=BAR_SECONDS)
bars_df.head()

## Inspect the OFI distribution

In [ ]:
ofi = bars_df["ofi"]
smoothed = ofi.rolling(SMOOTH_WINDOW).mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
bars_df["close"].plot(ax=axes[0], color="black", alpha=0.8)
axes[0].set_ylabel("Price")
axes[0].set_title(f"{SYMBOL} 1-minute close")
axes[0].grid(True, alpha=0.3)

smoothed.plot(ax=axes[1], color="steelblue", linewidth=0.8)
for level, color in [(ENTRY_THRESHOLD, "green"), (-ENTRY_THRESHOLD, "red"), (0, "black")]:
    axes[1].axhline(level, color=color, linestyle="--", linewidth=0.6, alpha=0.5)
axes[1].set_ylabel(f"OFI ({SMOOTH_WINDOW}-bar smoothed)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Run the event-driven backtest

In [ ]:
bars = bars_from_dataframe(bars_df[["open", "high", "low", "close", "volume"]])
ofi_lookup = ofi.to_dict()

strategy = RollingOFIStrategy(
    ofi_lookup=ofi_lookup, window=SMOOTH_WINDOW, threshold=ENTRY_THRESHOLD
)
result = run_event_backtest(
    bars,
    strategy,
    fee_bps=2.0,
    slippage_bps=5.0,
    periods_per_year=PERIODS_PER_YEAR,
)
pd.Series(result.metrics)

## Equity curve vs buy & hold

In [ ]:
bh = (1.0 + bars_df["close"].pct_change().fillna(0.0)).cumprod() * result.config["initial_capital"]

fig, ax = plt.subplots(figsize=(11, 5))
result.equity_curve.plot(ax=ax, label="OFI strategy")
bh.plot(ax=ax, label="Buy & hold", alpha=0.6)
ax.set_title(f"{SYMBOL} {START} \u2014 OFI strategy vs buy & hold (1-minute)")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()